## Ingestion of regression data from public sources
____
This is where we are going to collect all the features and targets we are going to use for the multimodal model. Our features include on the ground atmospheric data, satellite imagery for days with smoke from either industry or wildfires. The target will be on the ground readings of PM2.5 measures.

### 1. Gather and put all on-the-ground readings into a single table seperated by site

- CSV files downloaded from EPA AirData portal located in src/ingestion/raw_data
- Study Area: California
- Study period: 2018-01-01 - 2025-12-31
- _Source_: https://rconnect-public.epa.gov/AirData/#shiny-tab-DailySummary

In [2]:
import pandas as pd
from pathlib import Path
import requests
from config import settings

SOURCE_DIR = settings.raw_data

directory = Path(SOURCE_DIR)
df = []

for item in directory.glob("*.csv"):
    data = pd.read_csv(item)
    df.append(data)
df = pd.concat(df, ignore_index=True)
df.to_csv("combined_pm25_readings.csv") # GATE 1


### 2. Clean the data so there are no duplicates, no null values, and all the data types are the same throughout

- Some duplicates are from the same site on the same day but with different sensors. Will make the assumption that the most reliable one is the lowest.
- Others have unexplained duplicates that will be diagnosed

In [10]:
EPA_KEY = settings.epa_key
BASE_URL = settings.base_url
def test_suspicious_dates(date: str):    
    PARAMS = {
        "email": settings.epa_user,
        "key": settings.epa_key,
        "param": 88101,
        "state": "06", "county": "001", "site": "0007",
        "bdate": date.replace('-', ""), "edate": date.replace('-', ""),
    }
    mask = (df["Site ID"] == 60010007) & (df["Date"] == date)
    pm25_val = df.loc[mask, "Daily Mean PM2.5 Concentration"]
    livermore_api_data = requests.get(f"{BASE_URL}dailyData/bySite", params=PARAMS)
    print(f"--------------{date}--------------------")
    print("--------------API SCORES-----------------", end='\n')
    print([d["arithmetic_mean"] for d in livermore_api_data.json().get("Data", [])], end='\n')
    print("--------------DOWNLOAD DATA--------------", end='\n')
    print(f"Download Data PM2.5: {pm25_val.item()}", end='\n\n')

test_suspicious_dates("2018-01-01")
test_suspicious_dates("2018-01-02")
test_suspicious_dates("2018-01-03")
test_suspicious_dates("2018-01-04")
test_suspicious_dates("2018-01-05")

--------------2018-01-01--------------------
--------------API SCORES-----------------
[28.166667, 28.1, 28.1, 28.1, 28.1, 28.1, 28.1, 28.1, 28.1]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 24.5

--------------2018-01-02--------------------
--------------API SCORES-----------------
[24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5, 24.5]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 39.0

--------------2018-01-03--------------------
--------------API SCORES-----------------
[39.086957, 39.0, 39.0, 39.0, 39.0, 39.0, 39.0, 39.0, 39.0]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 20.9

--------------2018-01-04--------------------
--------------API SCORES-----------------
[20.916667, 20.9, 20.9, 20.9, 20.9, 20.9, 20.9, 20.9, 20.9]
--------------DOWNLOAD DATA--------------
Download Data PM2.5: 5.5

--------------2018-01-05--------------------
--------------API SCORES-----------------
[5.583333, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5, 5.5]
----

### ROADBLOCK

The previous experiment confirmed a worrying suspicion. The data collected through the AirData portal contains scores/measurements of the next day. The data was download with an extra day from the previous year and one less from the current year. The reasoning is clear now that this is because the data for each day is the measured data for the **next day's** measurements. This was tested against what the API provided for that same day and the same test was ran for the next 4 days after that one. On each day, the previous day's PM2.5 measure from the downloaded data was the current day's PM2.5 measure from the API. It can be assumed that the downloaded data is the faulty one as inherently it adds Dec 31st of the previous year to any year's range and omits that year's own Dec 31st, leading us to suspect, the data has an offset of one day.

### NEXT STEPS

Deciding between fetching the data from the API instead or using the downloaded data and fix the offset. This causes a dilemma however because if the measurements are for the wrong day, everything else probably is and might need to be fixed. All-in-all the downloaded data may be too unpredictable and somewhat of a hassle to work with and using the API may be a better option.

### PARSING THE DATA

So the API returns multiple rows for some sites for the same measurement day. The difference between these rows is the pollutant standard, the measuring device, the event exclusion/inclusion decision and sample duration. The pollutant standard does not matter in this use case but decisions have to be made about the others. Here is what I suggest:

- **Measuring device:** Always pick the continuous monitor and use its average value for that day and use the filter sampler as a fallback if it does not exist. If two continuos monitors exist
- **Event Inclusion/Exclusion:** Always use "Events Included" or "No Events" never "Concurred Events Excluded", this is where the truth lies for unnatural smoke days
- **Sample Duration:** Use the hourly sample monitor and its average for the data

With that the proposed attributes for each sample day and site will be:

- `state_code`
- `county_code`
- `site_number`
- `latitude`
- `longitude`
- `date_local`
- `observation_count`
- `observation_percent`
- `arithmetic_mean`
- `method_code`
- `poc`
- `event_type`
- `date_of_last_change`



In [ ]:
def get_california_pm25_data(year: str):
    PARAMS = {
            "email": settings.epa_user,
            "key": settings.epa_key,
            "param": 88101,
            "state": "06",
            "bdate": f"{year}-01-01", "edate": f"{year}-12-31",
        }

    api_data = requests.get(f"{BASE_URL}dailyData/byState", params=PARAMS)